# Meaning Directness Evaluator

**The Meaning Directness Evaluator** measures how explicit, literal, and straightforward a text's meaning is for students in grades 3-12. It assesses whether meaning is on the surface or requires abstract reasoning, interpretation of figurative language, or familiarity with unconventional expressions. When you run a passage through the evaluator, it returns a structured output that includes:

* **complexity_score**: The conventionality complexity level (`slightly_complex` to `exceedingly_complex`).
* **conventionality_features**: Specific language features driving the complexity (e.g., idioms, metaphors, implied meaning) with direct quotes from the text.
* **grade_context**: How the conventionality demands compare to general expectations for the target grade.
* **instructional_insights**: Actionable pedagogical suggestions for scaffolding the unconventional language features in the classroom.
* **reasoning**: A synthesis of why the text fits the chosen complexity level.

This gives you a clear signal about the figurative and abstract language demands of a passage, helping ensure AI-generated content is appropriate for the target grade.

Everything this notebook runs — the model, the temperature, the prompts, the output schema — is loaded from `config.json` in this directory. Nothing is hardcoded below, so the notebook cannot drift from the canonical assets.

In [ ]:
%pip install -qU langchain-google-genai langchain textstat python-dotenv

In [ ]:
import getpass
import hashlib
import json
import os
import pprint as pp
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
import textstat

In [ ]:
# Check for the API key
load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

### Load the canonical assets

`config.json` is the source of truth. The prompt files it declares are loaded from disk and
verified against the `sha256` recorded in the config — a drift tripwire. `scripts/check.py`
enforces the same hashes in CI. `input_schema.json` is a CI-side contract (`scripts/check.py`
binds each fixture's `input` to it) and isn't needed at runtime, so only `output_schema.json`
is loaded here — it drives structured output below.

In [ ]:
ASSETS_DIR = Path(".")

with open(ASSETS_DIR / "config.json") as f:
    CONFIG = json.load(f)

with open(ASSETS_DIR / "output_schema.json") as f:
    OUTPUT_SCHEMA = json.load(f)

# Load every prompt message declared in config, in order. Each message has
# {role, source_path, sha256}; a hash mismatch means the prompt on disk drifted
# from what the config pins, so we fail loudly rather than silently evaluate.
PROMPT_MESSAGES = []  # list of (role, text) tuples, preserving config order
for msg_spec in CONFIG["steps"][0]["prompt"]["messages"]:
    role = msg_spec["role"]
    raw = (ASSETS_DIR / msg_spec["source_path"]).read_bytes()
    text = raw.decode("utf-8")
    actual_sha = hashlib.sha256(raw).hexdigest()
    declared_sha = msg_spec["sha256"]
    assert actual_sha == declared_sha, (
        f"prompt drift detected for role={role!r} ({msg_spec['source_path']}): "
        f"declared {declared_sha[:12]}..., actual on disk {actual_sha[:12]}..."
    )
    PROMPT_MESSAGES.append((role, text))

_STEP = CONFIG["steps"][0]

print(f"Loaded {CONFIG['evaluator']['id']} from {ASSETS_DIR.resolve()}")
print(f"  model:       {_STEP['model']['name']}")
print(f"  temperature: {_STEP['generation']['temperature']}")
print(f"  levels:      {OUTPUT_SCHEMA['properties']['complexity_score']['enum']}")
print("  prompts:")
for msg_spec, (role, text) in zip(_STEP["prompt"]["messages"], PROMPT_MESSAGES):
    sha = hashlib.sha256((ASSETS_DIR / msg_spec["source_path"]).read_bytes()).hexdigest()[:12]
    print(f"    {role:>6}  {msg_spec['source_path']:<12} ({len(text):>5} chars, sha {sha})")

### FK score preprocessing

Declared as a `preprocessing` step in `config.json` and computed deterministically here, the same as Intertextuality/Organizational Structure/Purpose.

In [ ]:
_FK_PRE = next(p for p in CONFIG["preprocessing"] if p["id"] == "fk_score")
_FK_IMPL = _FK_PRE["implementation"]["python"]
_FK_LIB = _FK_IMPL["library"]
_FK_FN = _FK_IMPL["function"]
_FK_TRANSFORM = _FK_IMPL["post_transform"]

if _FK_LIB != "textstat":
    raise ValueError(f"unsupported fk library in config: {_FK_LIB!r}")


def calculate_fk_score(text) -> float:
    """Compute Flesch-Kincaid Grade Level per CONFIG['preprocessing']."""
    fn = getattr(textstat, _FK_FN)
    value = fn(text)
    if _FK_TRANSFORM["type"] == "round":
        value = round(value, _FK_TRANSFORM["precision"])
    else:
        raise ValueError(f"unsupported post_transform type: {_FK_TRANSFORM['type']!r}")
    return value

### Set up the Meaning Directness Evaluator function

In [ ]:
def evaluate_conventionality(text: str, grade: str):
    """
    Evaluate the Meaning Directness complexity of a text using the canonical config in
    this directory (config.json + system.txt + user.txt).

    Returns a dict with full I/O trace fields:
      - rendered_prompt:  the actual list of messages sent to the model (input-side trace)
      - raw_output:       the AIMessage returned by the LLM (keeps response/usage metadata)
      - raw_text:         just the string content of the AIMessage
      - formatted_output: the parsed dict matching OUTPUT_SCHEMA
      - usage:            token-usage metadata if the provider returned it

    The LLM is invoked ONCE; include_raw=True returns both the raw AIMessage and the
    parsed output without a second call.
    """
    # parser.kind == "structured_output" -> use the model's native output enforcement,
    # driven by output_schema.json rather than a prompt-side format instruction.
    llm = ChatGoogleGenerativeAI(
        model=_STEP["model"]["name"],
        temperature=_STEP["generation"]["temperature"],
    )
    structured_llm = llm.with_structured_output(OUTPUT_SCHEMA, include_raw=True)

    # Every message's content was loaded from disk and hash-verified above, so we can
    # feed the (role, text) tuples straight into the template.
    prompt_template = ChatPromptTemplate.from_messages(PROMPT_MESSAGES)

    try:
        fk_score = calculate_fk_score(text)
        inputs = {"text": text, "grade_level": grade, "fk_score": fk_score}
        rendered_messages = prompt_template.format_messages(**inputs)
        raw = structured_llm.invoke(rendered_messages)

        if raw.get("parsing_error"):
            raise ValueError(f"structured output parsing failed: {raw['parsing_error']}")

        return {
            "rendered_prompt": [m.model_dump() for m in rendered_messages],
            "raw_output": raw["raw"],
            "raw_text": raw["raw"].content,
            "formatted_output": raw["parsed"],
            "usage": getattr(raw["raw"], "usage_metadata", None),
        }
    except Exception as e:
        return f"Error evaluating text: {e}"

# Try evaluating text for conventionality complexity

Evaluator may take up to a minute to run. While the evaluator is running, you will see that the
status of the kernel is busy. For Jupyter, you can see the status of kernel at the bottom of the page.

To evaluate your own text, replace the text and grade below and run the cell again.

In [ ]:
# Swap out text and grade here for your own text to evaluate
sample_text = """
"Well, then," said the teacher, "you may take your slate and go out behind the schoolhouse for half an hour. Think of something to write about, and write the word on your slate. Then try to tell what it is, what it is like, what it is good for, and what is done with it. That is the way to write a composition." Henry took his slate and went out. Just behind the schoolhouse was Mr. Finney's barn. Quite close to the barn was a garden. And in the garden, Henry saw a turnip. "Well, I know what that is," he said to himself; and he wrote the word turnip on his slate. Then he tried to tell what it was like, what it was good for, and what was done with it. Before the half hour was ended he had written a very neat composition on his slate. He then went into the house, and waited while the teacher read it. The teacher was surprised and pleased. He said, "Henry Longfellow, you have done very well. Today you may stand up before the school and read what you have written about the turnip."
"""

result = evaluate_conventionality(sample_text, grade="4")

if isinstance(result, dict):
    out = result["formatted_output"]
    print(f"Complexity score: {out['complexity_score']}")
    print(f"Grade context:    {out['grade_context']}")
    print(f"\nConventionality features:")
    for feat in out["conventionality_features"]:
        print(f"  - {feat}")
    print(f"\nInstructional insights:\n{out['instructional_insights']}")
    print(f"\nReasoning:\n{out['reasoning']}")
else:
    print(result)

### Full I/O trace

In [ ]:
print("=" * 60)
print("RENDERED PROMPT (input sent to the LLM)")
print("=" * 60)
pp.pprint(result["rendered_prompt"])

print("\n" + "=" * 60)
print("RAW LLM TEXT (model's verbatim output)")
print("=" * 60)
print(result["raw_text"])

print("\n" + "=" * 60)
print("PARSED OUTPUT (output_schema)")
print("=" * 60)
pp.pprint(result["formatted_output"])

print("\n" + "=" * 60)
print("USAGE METADATA")
print("=" * 60)
pp.pprint(result["usage"])

### Sniff-test runner

Runs the cases in `fixtures.json` and compares the predicted complexity level against the
expected label. Per `config.json`'s `fixtures.tolerance`, a prediction one rubric step away
from the expected label also counts as a pass.

In [ ]:
fixtures_path = ASSETS_DIR / CONFIG["fixtures"]["path"]
with open(fixtures_path) as f:
    fixtures = json.load(f)
print(f"Loaded {len(fixtures)} fixtures from {fixtures_path.name}\n")

_RUBRIC_ORDER = OUTPUT_SCHEMA["properties"]["complexity_score"]["enum"]
_ALLOW_ADJ = bool(CONFIG["fixtures"]["tolerance"].get("allow_adjacent_levels", False))

def _score_outcome(predicted: str, expected: str):
    """Return ('exact' | 'adjacent' | 'fail', distance_or_None)."""
    if predicted == expected:
        return "exact", 0
    if _ALLOW_ADJ and predicted in _RUBRIC_ORDER and expected in _RUBRIC_ORDER:
        d = abs(_RUBRIC_ORDER.index(predicted) - _RUBRIC_ORDER.index(expected))
        if d == 1:
            return "adjacent", d
    return "fail", None

# Optional capture: set EVAL_CAPTURE_DIR to a directory to record the exact
# rendered request + raw response for each fixture case, alongside its
# formatted output. Not wired into any check yet -- raw material for the
# planned replay-based contract tests. Unset by default, and always unset in
# CI, so this never changes what actually runs.
_CAPTURE_DIR = os.environ.get("EVAL_CAPTURE_DIR")
captures = []

results = []
for fx in fixtures:
    expected = fx["expected"]["complexity_score"]
    out = evaluate_conventionality(text=fx["input"]["text"], grade=fx["input"]["grade_level"])
    if isinstance(out, str):  # error path
        results.append({"id": fx["id"], "status": "error", "predicted": None, "expected": expected, "error": out})
        continue
    predicted = out["formatted_output"]["complexity_score"]
    status, _ = _score_outcome(predicted, expected)
    results.append({
        "id": fx["id"], "status": status,
        "predicted": predicted, "expected": expected,
        "description": fx.get("description", ""),
    })
    if _CAPTURE_DIR:
        captures.append({
            "id": fx["id"],
            "rendered_prompt": out["rendered_prompt"],
            "raw_text": out["raw_text"],
            "formatted_output": out["formatted_output"],
            "model": _STEP["model"],
            "temperature": _STEP["generation"]["temperature"],
            "usage": out["usage"],
        })

print("=" * 78)
print(f"{'ID':>8}  {'STATUS':<8}  {'PREDICTED':<22}  {'EXPECTED':<22}  DESCRIPTION")
print("=" * 78)
for r in results:
    icon = {"exact": "PASS", "adjacent": "PASS*", "fail": "FAIL", "error": "ERR"}[r["status"]]
    print(f"{r['id']:>8}  {icon:<8}  {(r['predicted'] or '-'):<22}  {r['expected']:<22}  {r.get('description','')[:25]}")

n_total = len(results)
n_exact = sum(1 for r in results if r["status"] == "exact")
n_adj   = sum(1 for r in results if r["status"] == "adjacent")
n_fail  = sum(1 for r in results if r["status"] == "fail")
n_err   = sum(1 for r in results if r["status"] == "error")
print("=" * 78)
print(f"Summary: {n_exact} exact, {n_adj} adjacent (tolerated), {n_fail} fail, {n_err} error  --  total {n_total}")
if _ALLOW_ADJ:
    print("(Adjacency tolerance ON: predictions within +/-1 rubric step of the expected label count as PASS*.)")

if _CAPTURE_DIR:
    capture_path = Path(_CAPTURE_DIR) / "captures.json"
    capture_path.write_text(json.dumps(captures, indent=2) + "\n")
    print(f"\nWrote {len(captures)} captures to {capture_path.resolve()}")